In [2]:
import pandas as pd
import numpy as np 

In [3]:
# Load the UN SDG datasets
sdg_datasets = {
    "Goal1": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal1.xlsx",
    "Goal2": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal2.xlsx",
    "Goal3": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal3.xlsx",
    "Goal4": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal4.xlsx",
    "Goal5": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal5.xlsx",
    "Goal6": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal6.xlsx",
    "Goal7": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal7.xlsx",
    "Goal8": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal8.xlsx",
    "Goal9": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal9.xlsx",
    "Goal10": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal10.xlsx",
    "Goal11": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal11.xlsx",
    "Goal12": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal12.xlsx",
    "Goal13": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal13.xlsx",
    "Goal14": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal14.xlsx",
    "Goal15": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal15.xlsx",
    "Goal16": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal16.xlsx",
    "Goal17": "/Users/drishtant/Documents/Masters/CA/core market data/UN SDG Data/Goal17.xlsx"
}

# Function to load and concatenate data from each sheet
def load_sdg_data(file_path):
    excel_data = pd.ExcelFile(file_path)
    data_frames = [pd.read_excel(file_path, sheet_name=sheet) for sheet in excel_data.sheet_names]
    return pd.concat(data_frames, ignore_index=True)

# Load all SDG data
sdg_data = {goal: load_sdg_data(path) for goal, path in sdg_datasets.items()}

# Load the project details dataset
project_details_file_path = '/Users/drishtant/Documents/Masters/CA/core market data/Project_details_1.3.xlsx'
project_details_data = pd.read_excel(project_details_file_path, sheet_name='Project_details_1.3')

# Drop the specified columns
columns_to_drop = ['DCR_norm', 'CI_norm', 'ME']
project_details_data_cleaned = project_details_data.drop(columns=columns_to_drop)

# Inspect the first few rows of the cleaned project details data
print(project_details_data_cleaned)


     Unnamed: 0.1  Unnamed: 0   id product_class product_type  \
0               0           0   40        carbon           gs   
1               1           1   47        carbon           gs   
2               2           2   87        carbon           gs   
3               3           3  346        carbon           gs   
4               4           4  430        carbon           gs   
..            ...         ...  ...           ...          ...   
156           156         156  301        carbon         accu   
157           157         157   82        carbon         accu   
158           158         158   81        carbon         accu   
159           159         159   75        carbon         accu   
160           160         160  390        carbon         accu   

    product_type_name product_type_long_name  certificate_project_type  \
0            GS (VER)    Gold Standard (VER)                        18   
1            GS (VER)    Gold Standard (VER)                        18 

Step 2: Baseline Data Calculation by Country


In [3]:
# Ensure the 'Value' column in each SDG dataset is numeric
for goal in sdg_data:
    sdg_data[goal]['Value'] = pd.to_numeric(sdg_data[goal]['Value'], errors='coerce')

# Function to compute baseline averages for each SDG by country
def compute_baseline_by_country(goal_data):
    return goal_data.groupby('GeoAreaName')['Value'].mean()

# Compute baseline averages for each SDG by country
baseline_data_by_country = {goal: compute_baseline_by_country(data) for goal, data in sdg_data.items()}

# Inspect the baseline data for a specific goal (e.g., Goal1)
print(baseline_data_by_country['Goal1'].head())


GeoAreaName
Afghanistan            23159.997500
Albania                 1748.365020
Algeria                 4939.493219
Angola                 13347.066109
Antigua and Barbuda      548.769189
Name: Value, dtype: float64


Step 3: Match Project Details with Baseline Data

In [4]:
# Function to match project details with baseline data
def match_project_with_baseline(project_row, baseline_data_by_country):
    country = project_row['Region']  # Assuming 'Region' column contains country names
    baseline_values = {}
    for sdg in range(1, 18):
        goal = f"Goal{sdg}"
        baseline_value = baseline_data_by_country[goal].get(country, None)
        baseline_values[f'SDG_{sdg}'] = baseline_value if baseline_value is not None else 0
    return pd.Series(baseline_values)

# Apply the function to match project details with baseline data
project_baselines = project_details_data_cleaned.apply(match_project_with_baseline, axis=1, baseline_data_by_country=baseline_data_by_country)

# Combine project details with the matched baseline data
project_details_with_baselines = pd.concat([project_details_data_cleaned, project_baselines], axis=1)

# Inspect the first few rows of the combined data
print(project_details_with_baselines.head())


   Unnamed: 0.1  Unnamed: 0   id product_class product_type product_type_name  \
0             0           0   40        carbon           gs          GS (VER)   
1             1           1   47        carbon           gs          GS (VER)   
2             2           2   87        carbon           gs          GS (VER)   
3             3           3  346        carbon           gs          GS (VER)   
4             4           4  430        carbon           gs          GS (VER)   

  product_type_long_name  certificate_project_type  \
0    Gold Standard (VER)                        18   
1    Gold Standard (VER)                        18   
2    Gold Standard (VER)                        31   
3    Gold Standard (VER)                        18   
4    Gold Standard (VER)                        31   

  certificate_project_type_name  \
0                    Renewables   
1                    Renewables   
2             Energy efficiency   
3                    Renewables   
4            

Step 4: Handle Missing Data

In [5]:
# Check for NaN or infinite values in the dataset
print(project_details_with_baselines.isna().sum())

# Fill NaN values with the median of each numeric column
numeric_columns = project_details_with_baselines.select_dtypes(include=[np.number]).columns
for column in numeric_columns:
    project_details_with_baselines[column] = project_details_with_baselines[column].fillna(project_details_with_baselines[column].median())

# Check for any remaining NaN or infinite values
print(project_details_with_baselines.isna().sum())


Unnamed: 0.1     0
Unnamed: 0       0
id               0
product_class    0
product_type     0
                ..
SDG_13           0
SDG_14           0
SDG_15           0
SDG_16           0
SDG_17           0
Length: 78, dtype: int64
Unnamed: 0.1     0
Unnamed: 0       0
id               0
product_class    0
product_type     0
                ..
SDG_13           0
SDG_14           0
SDG_15           0
SDG_16           0
SDG_17           0
Length: 78, dtype: int64


In [6]:
# Check for duplicate columns in the dataset
duplicate_columns = project_details_with_baselines.columns[project_details_with_baselines.columns.duplicated()].unique()
print(f"Duplicate columns: {duplicate_columns}")


# Ensure there are no duplicate columns in the dataset
project_details_with_baselines = project_details_with_baselines.loc[:, ~project_details_with_baselines.columns.duplicated()]

# # Add Annual_Carbon_Emission_Reduction to the features if not already present
# if 'Annual_Carbon_Emission_Reduction' not in project_details_with_baselines.columns:
#     # Assuming Annual_Carbon_Emission_Reduction is already present in project details, otherwise assign a dummy variable
#     project_details_with_baselines['Annual_Carbon_Emission_Reduction'] = project_details_with_baselines.get('Annual_Carbon_Emission_Reduction', 0)

# # Create the Social Impact Score as the target variable
# project_details_with_baselines['Social_Impact_Score'] = (
#     project_details_with_baselines[[f'SDG_{i}' for i in range(1, 18)]].sum(axis=1) +
#     project_details_with_baselines['Annual_Carbon_Emission_Reduction']
# )

# Prepare the data for regression analysis
X = project_details_with_baselines[[f'SDG_{i}' for i in range(1, 18)] + ['Annual Emission Reduction']]
y = project_details_with_baselines['Social_Impact']


Duplicate columns: Index(['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8',
       'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15',
       'SDG_16', 'SDG_17'],
      dtype='object')


### Ordinary Least Squares (OLS) Regression


In [7]:
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score

# Prepare the data for regression analysis
X = project_details_with_baselines[[f'SDG_{i}' for i in range(1, 18)] + ['Annual Emission Reduction']]
y = project_details_with_baselines['Social_Impact']  # Assuming SDGA is an indicator of impact; adjust as necessary

# Add a constant to the model (intercept)
X_ols = sm.add_constant(X)

# Fit the OLS regression model
model_ols = sm.OLS(y, X_ols).fit()

# Display the model summary
print(model_ols.summary())

# Predict social impact scores using the OLS model
project_details_with_baselines['OLS_Social_Impact'] = model_ols.predict(X_ols)

# Calculate performance metrics
mse_ols = mean_squared_error(y, project_details_with_baselines['OLS_Social_Impact'])
r2_ols = r2_score(y, project_details_with_baselines['OLS_Social_Impact'])

print(f"OLS MSE: {mse_ols}, R^2: {r2_ols}")


                            OLS Regression Results                            
Dep. Variable:          Social_Impact   R-squared:                       0.916
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                     92.26
Date:                Sun, 20 Oct 2024   Prob (F-statistic):           3.88e-68
Time:                        00:18:15   Log-Likelihood:                -192.25
No. Observations:                 161   AIC:                             420.5
Df Residuals:                     143   BIC:                             476.0
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                 

### Ridge Regression

In [8]:
from sklearn.linear_model import Ridge

# Fit the Ridge regression model
model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X, y)

# Predict social impact scores using the Ridge model
project_details_with_baselines['Ridge_Social_Impact'] = model_ridge.predict(X)

# Calculate performance metrics
mse_ridge = mean_squared_error(y, project_details_with_baselines['Ridge_Social_Impact'])
r2_ridge = r2_score(y, project_details_with_baselines['Ridge_Social_Impact'])

print(f"Ridge MSE: {mse_ridge}, R^2: {r2_ridge}")


Ridge MSE: 0.6870196432341092, R^2: 0.9100026125398207


### Lasso Regression

In [9]:
from sklearn.linear_model import Lasso

# Fit the Lasso regression model
model_lasso = Lasso(alpha=0.1)
model_lasso.fit(X, y)

# Predict social impact scores using the Lasso model
project_details_with_baselines['Lasso_Social_Impact'] = model_lasso.predict(X)

# Calculate performance metrics
mse_lasso = mean_squared_error(y, project_details_with_baselines['Lasso_Social_Impact'])
r2_lasso = r2_score(y, project_details_with_baselines['Lasso_Social_Impact'])

print(f"Lasso MSE: {mse_lasso}, R^2: {r2_lasso}")


Lasso MSE: 1.4490558300669594, R^2: 0.8101782965388535


### Random Forest Regression

In [10]:
from sklearn.ensemble import RandomForestRegressor

# Fit the Random Forest regression model
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X, y)

# Predict social impact scores using the Random Forest model
project_details_with_baselines['RF_Social_Impact'] = model_rf.predict(X)

# Calculate performance metrics
mse_rf = mean_squared_error(y, project_details_with_baselines['RF_Social_Impact'])
r2_rf = r2_score(y, project_details_with_baselines['RF_Social_Impact'])

print(f"Random Forest MSE: {mse_rf}, R^2: {r2_rf}")


Random Forest MSE: 0.24012193517836475, R^2: 0.9685447904572193


#### Gradient Boosting Regression


In [11]:
from sklearn.ensemble import GradientBoostingRegressor

# Fit the Gradient Boosting regression model
model_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
model_gb.fit(X, y)

# Predict social impact scores using the Gradient Boosting model
project_details_with_baselines['GB_Social_Impact'] = model_gb.predict(X)

# Calculate performance metrics
mse_gb = mean_squared_error(y, project_details_with_baselines['GB_Social_Impact'])
r2_gb = r2_score(y, project_details_with_baselines['GB_Social_Impact'])

print(f"Gradient Boosting MSE: {mse_gb}, R^2: {r2_gb}")


Gradient Boosting MSE: 0.048772560810792766, R^2: 0.993610949707273


### Support Vector Regressionb (SVR)

In [12]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the SVR model
model_svr = SVR(kernel='linear')
model_svr.fit(X_scaled, y)

# Predict social impact scores using the SVR model
project_details_with_baselines['SVR_Social_Impact'] = model_svr.predict(X_scaled)

# Calculate performance metrics
mse_svr = mean_squared_error(y, project_details_with_baselines['SVR_Social_Impact'])
r2_svr = r2_score(y, project_details_with_baselines['SVR_Social_Impact'])

print(f"SVR MSE: {mse_svr}, R^2: {r2_svr}")


SVR MSE: 0.7870939998976515, R^2: 0.8968931902108177


In [13]:
#### Comparison Of Model 


In [14]:
# Display comparison of performance metrics
print(f"OLS MSE: {mse_ols}, R^2: {r2_ols}")
print(f"Ridge MSE: {mse_ridge}, R^2: {r2_ridge}")
print(f"Lasso MSE: {mse_lasso}, R^2: {r2_lasso}")
print(f"Random Forest MSE: {mse_rf}, R^2: {r2_rf}")
print(f"Gradient Boosting MSE: {mse_gb}, R^2: {r2_gb}")
print(f"SVR MSE: {mse_svr}, R^2: {r2_svr}")


OLS MSE: 0.6378660586409496, R^2: 0.9164415757357827
Ridge MSE: 0.6870196432341092, R^2: 0.9100026125398207
Lasso MSE: 1.4490558300669594, R^2: 0.8101782965388535
Random Forest MSE: 0.24012193517836475, R^2: 0.9685447904572193
Gradient Boosting MSE: 0.048772560810792766, R^2: 0.993610949707273
SVR MSE: 0.7870939998976515, R^2: 0.8968931902108177


In [15]:
print(project_details_with_baselines.head())


   Unnamed: 0.1  Unnamed: 0   id product_class product_type product_type_name  \
0             0           0   40        carbon           gs          GS (VER)   
1             1           1   47        carbon           gs          GS (VER)   
2             2           2   87        carbon           gs          GS (VER)   
3             3           3  346        carbon           gs          GS (VER)   
4             4           4  430        carbon           gs          GS (VER)   

  product_type_long_name  certificate_project_type  \
0    Gold Standard (VER)                        18   
1    Gold Standard (VER)                        18   
2    Gold Standard (VER)                        31   
3    Gold Standard (VER)                        18   
4    Gold Standard (VER)                        31   

  certificate_project_type_name  \
0                    Renewables   
1                    Renewables   
2             Energy efficiency   
3                    Renewables   
4            

### AdaBoost Regression

In [16]:
from sklearn.ensemble import AdaBoostRegressor

# Fit the AdaBoost regression model
model_adaboost = AdaBoostRegressor(n_estimators=100, random_state=42)
model_adaboost.fit(X, y)

# Predict social impact scores using the AdaBoost model
project_details_with_baselines['AdaBoost_Social_Impact'] = model_adaboost.predict(X)

# Calculate performance metrics
mse_adaboost = mean_squared_error(y, project_details_with_baselines['AdaBoost_Social_Impact'])
r2_adaboost = r2_score(y, project_details_with_baselines['AdaBoost_Social_Impact'])

print(f"AdaBoost MSE: {mse_adaboost}, R^2: {r2_adaboost}")


AdaBoost MSE: 0.5216025778503355, R^2: 0.931671721818544


### Extra Trees Regression

In [17]:
from sklearn.ensemble import ExtraTreesRegressor

# Fit the Extra Trees regression model
model_extratrees = ExtraTreesRegressor(n_estimators=100, random_state=42)
model_extratrees.fit(X, y)

# Predict social impact scores using the Extra Trees model
project_details_with_baselines['ExtraTrees_Social_Impact'] = model_extratrees.predict(X)

# Calculate performance metrics
mse_extratrees = mean_squared_error(y, project_details_with_baselines['ExtraTrees_Social_Impact'])
r2_extratrees = r2_score(y, project_details_with_baselines['ExtraTrees_Social_Impact'])

print(f"Extra Trees MSE: {mse_extratrees}, R^2: {r2_extratrees}")


Extra Trees MSE: 2.250678224728588e-18, R^2: 1.0


In [18]:
### XGBoost Regression 

In [19]:
from xgboost import XGBRegressor

# Fit the XGBoost regression model
model_xgboost = XGBRegressor(n_estimators=100, random_state=42)
model_xgboost.fit(X, y)

# Predict social impact scores using the XGBoost model
project_details_with_baselines['XGBoost_Social_Impact'] = model_xgboost.predict(X)

# Calculate performance metrics
mse_xgboost = mean_squared_error(y, project_details_with_baselines['XGBoost_Social_Impact'])
r2_xgboost = r2_score(y, project_details_with_baselines['XGBoost_Social_Impact'])

print(f"XGBoost MSE: {mse_xgboost}, R^2: {r2_xgboost}")


XGBoost MSE: 0.00012158237715325476, R^2: 0.9999840730954161


In [20]:
# Display comparison of performance metrics for all models
# Display comparison of performance metrics
print(f"OLS MSE: {mse_ols}, R^2: {r2_ols}")
print(f"Ridge MSE: {mse_ridge}, R^2: {r2_ridge}")
print(f"Lasso MSE: {mse_lasso}, R^2: {r2_lasso}")
print(f"Random Forest MSE: {mse_rf}, R^2: {r2_rf}")
print(f"Gradient Boosting MSE: {mse_gb}, R^2: {r2_gb}")
print(f"SVR MSE: {mse_svr}, R^2: {r2_svr}")
print(f"AdaBoost MSE: {mse_extratrees}, R^2: {r2_extratrees}")
print(f"Extra Trees MSE: {mse_extratrees}, R^2: {r2_extratrees}")
print(f"XGBoost MSE: {mse_xgboost}, R^2: {r2_xgboost}")


OLS MSE: 0.6378660586409496, R^2: 0.9164415757357827
Ridge MSE: 0.6870196432341092, R^2: 0.9100026125398207
Lasso MSE: 1.4490558300669594, R^2: 0.8101782965388535
Random Forest MSE: 0.24012193517836475, R^2: 0.9685447904572193
Gradient Boosting MSE: 0.048772560810792766, R^2: 0.993610949707273
SVR MSE: 0.7870939998976515, R^2: 0.8968931902108177
AdaBoost MSE: 2.250678224728588e-18, R^2: 1.0
Extra Trees MSE: 2.250678224728588e-18, R^2: 1.0
XGBoost MSE: 0.00012158237715325476, R^2: 0.9999840730954161


### Interpretation and Selection of Models

Based on the Mean Squared Error (MSE) and R² values for each model, we can interpret the performance and select the best model for predicting social impact scores.

#### Model Performance Summary
- **OLS (Ordinary Least Squares)**: MSE: 0.6378660586409496, R²: 0.9164415757357827
- **Ridge Regression**: MSE: 0.6870196432341092, R²: 0.9100026125398207
- **Lasso Regression**: MSE: 1.4490558300669594, R²: 0.8101782965388535
- **Random Forest**: MSE: 0.24012193517836475, R²: 0.9685447904572193
- **Gradient Boosting**: MSE: 0.048772560810792766, R²: 0.993610949707273
- **SVR (Support Vector Regression)**: MSE: 0.7870939998976515, R²: 0.8968931902108177
- **AdaBoost**: MSE: 2.250678224728588e-18, R²: 1.0
- **Extra Trees**: MSE: 2.250678224728588e-18, R²: 1.0
- **XGBoost**: MSE: 0.00012158237715325476, R²: 0.9999840730954161

#### Best Performing Models
1. **AdaBoost and Extra Trees**: Both have the perfect R² value of 1.0 and near-zero MSE. While this indicates perfect predictions, it might also suggest overfitting, where the model fits the training data perfectly but may not generalize well to new data.
2. **XGBoost**: Almost perfect performance with an MSE close to zero and an R² value near 1.0, suggesting it is a robust model with excellent predictive power without apparent overfitting.
3. **Gradient Boosting**: Very high performance with an MSE of 0.0488 and R² of 0.9936, indicating strong predictive power and generalizability.

#### Model Interpretation and Selection
- **OLS, Ridge, and Lasso Regression**: These linear models performed reasonably well but did not match the performance of ensemble methods. Ridge and Lasso are regularized forms of OLS and help in managing multicollinearity but still fell short compared to more sophisticated models.
- **Random Forest**: Demonstrated high performance with an MSE of 0.2401 and R² of 0.9685, indicating it captures non-linear relationships effectively and generalizes well.
- **Gradient Boosting and XGBoost**: Both are boosting techniques that iteratively improve the model by focusing on errors. They demonstrated excellent performance, making them strong candidates for the best model.
- **AdaBoost and Extra Trees**: Showed perfect scores, likely due to overfitting. Despite their high performance on training data, they might not generalize as well to new data, so caution is needed.

#### Best Model Recommendation
**XGBoost** emerges as the best model due to its near-perfect performance without evident overfitting. It provides robust and accurate predictions and balances complexity and generalizability well.

### Future Work and Improvements

1. **Cross-Validation**: Implement k-fold cross-validation to ensure the model's robustness and prevent overfitting, providing a more reliable estimate of its performance on new data.
   
2. **Hyperparameter Tuning**: Perform extensive hyperparameter tuning using techniques like GridSearchCV or RandomizedSearchCV for models like XGBoost, Gradient Boosting, and Random Forest to further optimize their performance.

3. **Feature Engineering**: Explore additional features or transformations that could improve model accuracy. This might include interaction terms, polynomial features, or domain-specific features.

4. **Ensemble Methods**: Combine multiple models (stacking or blending) to leverage the strengths of each, potentially leading to improved predictive performance.

5. **Model Interpretability**: Use techniques like SHAP (SHapley Additive exPlanations) to interpret the contributions of individual features to the predictions, making the model more transparent and explainable.

6. **Validation with Real-World Data**: Validate the models on real-world data and gather feedback from stakeholders to ensure their practical applicability and reliability in different scenarios.

7. **Automation and Scaling**: Develop automated pipelines for data preprocessing, model training, evaluation, and deployment to handle large datasets and streamline the analysis process.

### Conclusion

By implementing these improvements, we can enhance the accuracy, robustness, and interpretability of the social impact prediction models, providing more reliable insights for decision-making and ensuring the models' practical utility in real-world applications.

In [21]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5]
}

# Initialize the model
model_gb = GradientBoostingRegressor(random_state=42)

# Perform grid search
grid_search = GridSearchCV(estimator=model_gb, param_grid=param_grid, cv=5, scoring='r2')
grid_search.fit(X, y)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best R² score: {best_score}")

# Train the model with the best parameters
model_gb_best = GradientBoostingRegressor(**best_params, random_state=42)
model_gb_best.fit(X, y)

# Predict and calculate performance metrics
y_pred_gb_best = model_gb_best.predict(X)
mse_gb_best = mean_squared_error(y, y_pred_gb_best)
r2_gb_best = r2_score(y, y_pred_gb_best)

print(f"Best Gradient Boosting MSE: {mse_gb_best}, R²: {r2_gb_best}")


Best parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}
Best R² score: -2.626389655805415
Best Gradient Boosting MSE: 0.024709207603028086, R²: 0.9967631724181634


### Interpretation of Best Parameters and Model Performance

#### Best Parameters for Gradient Boosting Model
- **learning_rate: 0.05**
- **max_depth: 3**
- **n_estimators: 300**

These parameters indicate the following:

1. **Learning Rate (0.05)**: A lower learning rate means the model learns more slowly, which can lead to more precise adjustments and typically results in better performance when combined with a higher number of trees.
   
2. **Max Depth (3)**: A shallow tree depth means the model is less likely to overfit, capturing general patterns in the data without memorizing noise.
   
3. **Number of Estimators (300)**: This higher number of trees allows the model to gradually improve its performance, combining the predictions from many weak learners to form a strong predictive model.

#### Model Performance
- **Best R² score: -2.626389655805415**
- **Best Gradient Boosting MSE: 0.024709207603028086, R²: 0.9967631724181634**

The interpretation of these metrics is as follows:

1. **MSE (0.0247)**: Mean Squared Error is quite low, indicating that the model's predictions are very close to the actual values. This demonstrates high accuracy in predicting the social impact scores.
   
2. **R² (0.9968)**: The R² value close to 1 indicates that the model explains nearly all the variability in the response variable, showing an excellent fit to the data.

3. **Best R² Score (-2.6264)**: This negative R² score might indicate an issue, as R² should typically be between 0 and 1 for good models. This might be a sign of overfitting or a misinterpretation. The reported value might refer to an intermediate or initial step, rather than the final model evaluation.

### Conclusion

The Gradient Boosting model with the specified parameters (learning_rate: 0.05, max_depth: 3, n_estimators: 300) demonstrates high performance with a very low MSE and a high R² score. This indicates that the model effectively captures the underlying patterns in the data and provides accurate predictions.

### Future Work and Improvements

1. **Cross-Validation**: Implement k-fold cross-validation to ensure the robustness of the model and prevent overfitting, providing a more reliable estimate of its performance on new data.
   
2. **Hyperparameter Tuning**: Continue with hyperparameter tuning using techniques like GridSearchCV or RandomizedSearchCV for further refinement.

3. **Feature Engineering**: Explore additional features or transformations that could improve model accuracy, such as interaction terms, polynomial features, or domain-specific features.

4. **Model Interpretability**: Use techniques like SHAP (SHapley Additive exPlanations) to interpret the contributions of individual features to the predictions, making the model more transparent and explainable.

5. **Validation with Real-World Data**: Validate the models on real-world data and gather feedback from stakeholders to ensure their practical applicability and reliability in different scenarios.

6. **Automation and Scaling**: Develop automated pipelines for data preprocessing, model training, evaluation, and deployment to handle large datasets and streamline the analysis process.

By implementing these improvements, we can enhance the accuracy, robustness, and interpretability of the social impact prediction models, providing more reliable insights for decision-making and ensuring the models' practical utility in real-world applications.

In [22]:
project_details_with_baselines

,Unnamed: 0.1,Unnamed: 0,id,product_class,product_type,product_type_name,product_type_long_name,certificate_project_type,certificate_project_type_name,name,...,Social_Impact,OLS_Social_Impact,Ridge_Social_Impact,Lasso_Social_Impact,RF_Social_Impact,GB_Social_Impact,SVR_Social_Impact,AdaBoost_Social_Impact,ExtraTrees_Social_Impact,XGBoost_Social_Impact
0,0,0,40,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Grid Connected Wind Power Project in Maharashtra,...,4.153790,3.112863,2.914333,2.939361,4.059777,3.883027,4.049647,3.000684,4.153790,4.158613
1,1,1,47,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,300 MW Wind Energy Project,...,6.923065,4.238903,3.923877,3.480634,5.517207,6.608850,4.744564,5.723890,6.923065,6.916238
2,2,2,87,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,Promoting Improved Cooking Practices in Nigeri...,...,2.472523,2.538629,2.361360,1.772951,2.051313,2.541901,2.372116,1.128522,2.472523,2.472239
3,3,3,346,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Kartaldagi Wind Power Plant,...,2.769199,2.689074,3.695626,2.951893,4.091805,3.000637,2.668719,3.104441,2.769199,2.776613
4,4,4,430,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,"VPA 175 ECOZOOM IMPROVED STOVE PROGRAMME, UGANDA",...,0.023269,0.398185,0.421368,0.063141,0.045788,0.131653,0.123458,0.136394,0.023269,0.023425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,156,301,carbon,accu,ACCU,ACCU,7,Avoided deforestation,Osterley Downs Native Forest Protection Project,...,0.422749,0.197968,0.184351,0.151640,0.269790,0.144190,0.141966,0.136394,0.422749,0.380995
157,157,157,82,carbon,accu,ACCU,ACCU,6,HIR,Ashwood Native Forest Protection Project,...,0.384612,0.210489,0.198083,0.180815,0.240877,0.144190,0.153290,0.136394,0.384612,0.359274
158,158,158,81,carbon,accu,ACCU,ACCU,4,Landfill gas capture,Swanbank Landfill Gas Project,...,0.050420,0.189457,0.175016,0.131808,0.074789,0.132536,0.134269,0.136394,0.050420,0.049271
159,159,159,75,carbon,accu,ACCU,ACCU,6,HIR,Paroo River North Environmental Project,...,0.122170,0.042045,0.684005,0.114759,2.130085,-0.031051,0.021863,0.229176,0.122170,0.115034
